# AMTB Dose-Response Analysis: Lipid Droplet Measurements

### Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp

In [ ]:
# Load data
df = pd.read_csv('Dosage_Combined.csv')
 
# Tidy column names
df.columns = df.columns.str.strip().str.replace(' ', '_')
 
# Drop redundant column: RawIntDen is perfectly correlated with IntDen
df = df.drop(columns=['RawIntDen'])
 
# Set dosage order so plots and tables come out in the right sequence
dosage_order = ['Control', '0.5uM', '1uM', '3uM', '5uM', '10uM']
df['Dosage'] = pd.Categorical(df['Dosage'], categories=dosage_order, ordered=True)
 
params = ['Area', 'Perimeter', 'IntDen']
 
print("Data loaded successfully.")
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nFirst 5 rows:\n{df.head()}")

### Statistical Tests

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS (mean ± SD, n per group)")
print("="*60)
 
summary = df.groupby('Dosage')[params].agg(['mean', 'std', 'count'])
print(summary.round(2))

In [ ]:
# Check Normality: Shapiro-Wilk

""" 
Shapiro-Wilk: p < 0.05 means the data is NOT normally distributed
With large n, this test is very sensitive — even small deviations are flagged
All groups fail normality here, so we use non-parametric tests throughout
"""
 
print("\n" + "="*60)
print("NORMALITY CHECK: Shapiro-Wilk p-values")
print("p < 0.05 → NOT normal → use non-parametric tests")
print("="*60)
 
normality = {}
for dose in dosage_order:
    group = df[df['Dosage'] == dose]
    normality[dose] = {
        param: stats.shapiro(group[param])[1]
        for param in params
    }
 
norm_df = pd.DataFrame(normality).T  # rows = doses, cols = params
print(norm_df.map(lambda p: f"{p:.2e}"))

In [ ]:
# Check Variance Equality: Levene's test

"""
Levene's test: p < 0.05 → variances are NOT equal across groups
This confirms non-parametric approach is appropriate
"""
 
print("\n" + "="*60)
print("VARIANCE CHECK: Levene's test p-values")
print("p < 0.05 → unequal variances across dosage groups")
print("="*60)
 
for param in params:
    groups = [df[df['Dosage'] == d][param].values for d in dosage_order]
    _, p = stats.levene(*groups)
    result = "unequal variances ✗" if p < 0.05 else "equal variances ✓"
    print(f"  {param}: p = {p:.3e}  →  {result}")

In [ ]:
# Kruskal-Wallis Test

"""
Non-parametric equivalent of one-way ANOVA
H₀: all groups come from the same distribution
p < 0.05 → at least one group is significantly different
"""
 
print("\n" + "="*60)
print("KRUSKAL-WALLIS TEST")
print("H₀: no difference across dosage groups")
print("="*60)
 
kw_results = {}
for param in params:
    groups = [df[df['Dosage'] == d][param].values for d in dosage_order]
    H, p = stats.kruskal(*groups)
    kw_results[param] = (H, p)
    sig = "significant ✓" if p < 0.05 else "not significant"
    print(f"  {param}: H = {H:.2f}, p = {p:.3e}  →  {sig}")

In [ ]:
# Dunn's Post-Hoc Test - Bonferroni correction

"""
Pairwise comparisons after a significant Kruskal-Wallis result
Bonferroni correction controls for false positives across multiple comparisons
p < 0.05 after correction → those two groups are significantly different
"""
 
print("\n" + "="*60)
print("DUNN'S POST-HOC TEST (Bonferroni-corrected p-values)")
print("p < 0.05 → significant pairwise difference")
print("="*60)
 
dunn_results = {}
for param in params:
    dunn = sp.posthoc_dunn(
        df, val_col=param, group_col='Dosage', p_adjust='bonferroni'
    )
    dunn_results[param] = dunn
    print(f"\n{param}:")
    print(dunn.round(4))

### Visualizations

In [ ]:
palette = sns.color_palette("muted", n_colors=len(dosage_order))

In [ ]:
# Boxplot + strip for each parameter
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Lipid Droplet Measurements by AMTB Dose', fontsize=14, fontweight='bold')
 
for ax, param in zip(axes, params):
    sns.boxplot(
        x='Dosage', y=param, hue='Dosage', data=df,
        order=dosage_order, palette=palette, legend=False,
        width=0.5, flierprops=dict(marker='', alpha=0),  # hide default fliers
        ax=ax
    )
    sns.stripplot(
        x='Dosage', y=param, data=df,
        order=dosage_order, color='black',
        alpha=0.25, size=1.5, jitter=True, ax=ax
    )
    ax.set_title(param, fontsize=12)
    ax.set_xlabel('AMTB Dose')
    ax.set_ylabel(param)
    ax.tick_params(axis='x', rotation=30)
 
plt.tight_layout()
plt.savefig('boxplots_by_dose.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: boxplots_by_dose.png")

In [ ]:
# Violin + strip plot (shows distribution shape more clearly)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribution of Lipid Droplet Features by AMTB Dose',
             fontsize=14, fontweight='bold')
 
for ax, param in zip(axes, params):
    sns.violinplot(
        x='Dosage', y=param, hue='Dosage', data=df,
        order=dosage_order, palette=palette, legend=False,
        inner=None, ax=ax
    )
    sns.stripplot(
        x='Dosage', y=param, data=df,
        order=dosage_order, color='black',
        alpha=0.2, size=1.2, jitter=True, ax=ax
    )
    ax.set_title(param, fontsize=12)
    ax.set_xlabel('AMTB Dose')
    ax.set_ylabel(param)
    ax.tick_params(axis='x', rotation=30)
 
plt.tight_layout()
plt.savefig('violins_by_dose.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: violins_by_dose.png")

In [ ]:
# Mean ± SEM line plot (shows overall trend across doses)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Mean ± SEM by AMTB Dose', fontsize=13, fontweight='bold')
 
# Assign a numeric x-axis position for plotting
dose_positions = {d: i for i, d in enumerate(dosage_order)}
 
for ax, param in zip(axes, params):
    means = df.groupby('Dosage')[param].mean()
    sems  = df.groupby('Dosage')[param].sem()
    x_pos = [dose_positions[d] for d in means.index]
 
    ax.errorbar(x_pos, means.values, yerr=sems.values,
                fmt='o-', color='steelblue', capsize=4, linewidth=1.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(dosage_order, rotation=30)
    ax.set_title(param, fontsize=11)
    ax.set_xlabel('AMTB Dose')
    ax.set_ylabel(f'Mean {param}')
 
plt.tight_layout()
plt.savefig('mean_sem_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: mean_sem_trend.png")
 

In [ ]:
# Dunn's post-hoc heatmaps (visual summary of pairwise significance)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Dunn's Post-Hoc p-values (Bonferroni)\n* = p < 0.05",
             fontsize=13, fontweight='bold')
 
for ax, param in zip(axes, params):
    dunn = dunn_results[param]
    # Mask the upper triangle and diagonal for a cleaner look
    mask = np.triu(np.ones_like(dunn, dtype=bool))
    annot = dunn.map(lambda p: f"{p:.3f}\n*" if p < 0.05 else f"{p:.3f}")
    sns.heatmap(
        dunn, mask=mask, annot=annot, fmt='', cmap='RdYlGn_r',
        vmin=0, vmax=1, linewidths=0.5,
        ax=ax, cbar=ax == axes[-1]  # only show colorbar on last panel
    )
    ax.set_title(param, fontsize=11)
    ax.tick_params(axis='x', rotation=40)
    ax.tick_params(axis='y', rotation=0)
 
plt.tight_layout()
plt.savefig('dunn_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dunn_heatmaps.png")